# Pipeline de tratamento dos dados das ovitrampas (2024, 2025 e 2026)

In [1]:
import re
import unicodedata
import pandas
import xlrd  # necessário para ler os arquivos .xls de 2024 e 2025

In [2]:
# Pasta base dos dados e caminhos dos arquivos
BASE = r"C:\Users\joaoc\Downloads\PROJETO5"

path_2026 = BASE + r"\INFORMAÇÕES - GEVACZ\OVITRAMPAS 2026\PLANILHA GERAL OVT 2026 ATÃ_ O CICLO  - ESSA É A NOVA PLNILHA DE DIGITAÇÃO DAS OVITRAMPAS.xlsx"
path_antigos = {
    2025: BASE + r"\INFORMAÇÕES - GEVACZ\OVITRAMPAS 2025\Planilha OVT DS-{ds}.xls",
    2024: BASE + r"\INFORMAÇÕES - GEVACZ\OVITRAMPAS 2024\Planilha OVT DS-{ds}.xls",
}

DSS = ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII']
DB_NAME = "gevacz.db"


## 2026

In [3]:
# Coloca dados em uma DataFrame
ovitrampas2026 = pandas.read_excel(path_2026, 0, header=0)
ovitrampas2026.head()

,Ano,CICLOS,DS,BAIRRO,BAIRRO_AREA,AGENTE / MATRICULA,QT,ID_OVT,DT_COLETA,DT_ENTREGA_APOIO,DT_ENTREGA_LAB,DT_LEITURA,NM_TEC_LAB,DT_DIG,DT_ENVIO_DS,N_OVOS,OBS\nF (Fechado); E (Extraviado); R (Recusado); D (Desocupado); Rec (Recuperada)
0,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,1,REC1001,2026-01-29,2026-01-30,2026-02-05,2026-02-07,ANA CAROLINA,2026-03-07 00:00:00,NaN,16,NaN
1,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,1,REC1002,2026-01-29,2026-01-30,2026-02-05,2026-02-07,ANA CAROLINA,2026-03-07 00:00:00,NaN,14,NaN
2,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,1,REC1003,2026-01-29,2026-01-30,2026-02-05,2026-02-07,ANA CAROLINA,2026-03-07 00:00:00,NaN,83,NaN
3,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,1,REC1004,2026-01-29,2026-01-30,2026-02-05,2026-02-07,ANA CAROLINA,2026-03-07 00:00:00,NaN,77,NaN
4,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,1,REC1005,2026-01-29,2026-01-30,2026-02-05,2026-02-07,ANA CAROLINA,2026-03-07 00:00:00,NaN,88,NaN


In [4]:
# Remove colunas que não serão usadas (errors='ignore' permite reexecutar a célula sem dar KeyError)
col = ['QT','DT_COLETA','DT_ENTREGA_APOIO','DT_ENTREGA_LAB','DT_LEITURA','NM_TEC_LAB','DT_DIG','DT_ENVIO_DS']
ovitrampas2026 = ovitrampas2026.drop(columns=col, errors='ignore')
ovitrampas2026.head()

,Ano,CICLOS,DS,BAIRRO,BAIRRO_AREA,AGENTE / MATRICULA,ID_OVT,N_OVOS,OBS\nF (Fechado); E (Extraviado); R (Recusado); D (Desocupado); Rec (Recuperada)
0,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,REC1001,16,NaN
1,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,REC1002,14,NaN
2,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,REC1003,83,NaN
3,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,REC1004,77,NaN
4,2026,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,REC1005,88,NaN


## 2024 e 2025

In [5]:
import re
import unicodedata
import pandas


def _norm_nome(s):
    """Maiúsculas, sem acento e sem espaços extras (para comparar nomes)."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).strip().upper()


_RE_MAT = re.compile(r"\d{2,3}\.\d{3}-\d")      # matrícula no padrão 77.382-0


def _tem_matricula(txt):
    """True se o texto já traz uma matrícula (>= 5 dígitos, mesmo mal formatada, ex.: 772623 ou 77.376.3)."""
    return len(re.findall(r"\d", str(txt))) >= 5


def _nome_agente(texto):
    """Parte do texto que é o nome (remove matrícula, 'S/M' e barras)."""
    t = _RE_MAT.sub("", str(texto))
    t = re.sub(r"S\s*/?\s*M\b", "", t, flags=re.I)
    return _norm_nome(t.replace("/", " "))


def mapa_matriculas(ref):
    """Nome -> texto 'NOME/MATRICULA' de 2026, só para nomes com UMA matrícula."""
    achados = {}
    for txt in ref["AGENTE / MATRICULA"].dropna().astype(str).str.strip().unique():
        m = _RE_MAT.search(txt)
        if m:
            achados.setdefault(_nome_agente(txt), {})[m.group()] = txt
    return {n: list(v.values())[0] for n, v in achados.items() if len(v) == 1}


def completar_matricula(df, ref):
    """Para agentes sem matrícula, usa a de 2026 se o nome for igual (e inequívoco)."""
    mapa = mapa_matriculas(ref)
    ag = df["AGENTE / MATRICULA"].astype("string")
    sem = ag.notna() & ~ag.map(lambda t: _tem_matricula(t) if pandas.notna(t) else True).astype(bool)
    novo = ag.where(~sem, ag.map(lambda t: mapa.get(_nome_agente(t)) if pandas.notna(t) else None))
    df = df.copy()
    df["AGENTE / MATRICULA"] = novo.fillna(ag).astype(object)
    return df


def tratar_ovt_antigo(path, ds, ano, ref=None):
    """Lê uma planilha OVT antiga (2024/2025: uma aba por bairro, ciclos em colunas)
    e devolve um DataFrame longo no formato da planilha de 2026.

    ref: DataFrame de 2026, usado para BAIRRO / BAIRRO_AREA (pelo ID_OVT) e para
         completar a matrícula dos agentes pelo nome.
    """
    abas = pandas.read_excel(path, sheet_name=None, header=None)
    partes = []

    for nome_aba, raw in abas.items():
        # 1) linha do cabeçalho (a que tem 'BAIRRO' na 1ª coluna)
        hdr = raw.index[raw[0].astype(str).str.strip().eq("BAIRRO")][0]
        cab = raw.loc[hdr].astype(str).str.strip()

        # 2) coluna do ID (a posição varia entre abas) e linhas de ovitrampa.
        #    O ID pode ter 2 a 4 letras e barra (ROS/2001, ZU4001); descarta totais etc.
        c_id = cab[cab.str.upper().str.startswith("ID")].index[0]
        corpo = raw.loc[hdr + 1:]
        corpo = corpo[corpo[c_id].astype(str).str.strip().str.match(r"^[A-Za-z]{2,4}\s*/?\s*\d+$")]
        ids = corpo[c_id].astype(str).str.replace(r"[\s/]", "", regex=True).str.upper()

        # 3) cada ciclo = 3 colunas (DATA, NÚMERO OVOS, OBS). A numeração vem da linha
        #    de rótulos "CICLO n" (acima do cabeçalho), mais confiável do que contar 'DATA'.
        rot = raw.loc[hdr - 1].fillna("").astype(str).str.strip().str.upper()
        ciclos = {c: int(re.search(r"\d+", t).group()) for c, t in rot.items()
                  if re.match(r"^CICLO\s*\d+", t)}
        for c, n in ciclos.items():
            c_obs = c + 2 if c + 2 in raw.columns else None
            partes.append(pandas.DataFrame({
                "Ano": ano,
                "CICLOS": n,
                "DS": ds,
                "BAIRRO": nome_aba.strip().upper(),
                "AGENTE / MATRICULA": corpo[1].astype("string").str.strip().values,
                "ID_OVT": ids.values,
                "N_OVOS": pandas.to_numeric(corpo[c + 1], errors="coerce").values,
                "OBS": (corpo[c_obs].astype(str).str.strip().str.upper().values
                        if c_obs is not None else None),
                "DATA": pandas.to_datetime(corpo[c], errors="coerce").values,
            }))

    df = pandas.concat(partes, ignore_index=True)
    df["OBS"] = df["OBS"].replace({"NAN": pandas.NA, "NONE": pandas.NA, "": pandas.NA})
    # OBS: só códigos válidos (descarta lixo de digitação como '+', '.', 'P  -')
    df.loc[~df["OBS"].isin(["E", "F", "REC", "R", "D"]), "OBS"] = pandas.NA

    # 3b) linhas repetidas na planilha (mesma ovitrampa/ciclo/valores; uma delas sem agente)
    df = (df.sort_values("AGENTE / MATRICULA", na_position="last", kind="stable")
            .drop_duplicates(["ID_OVT", "CICLOS", "N_OVOS", "OBS", "DATA"], keep="first")
            .sort_index())

    # 4) descarta ciclos sem nenhum registro (sem data, sem ovos e sem observação)
    df = df[~(df["DATA"].isna() & df["N_OVOS"].isna() & df["OBS"].isna())]

    # 5) BAIRRO / BAIRRO_AREA: usa o de 2026 quando o ID existir lá
    df["BAIRRO_AREA"] = df["BAIRRO"]
    if ref is not None:
        mapa = ref.drop_duplicates("ID_OVT").set_index("ID_OVT")[["BAIRRO", "BAIRRO_AREA"]]
        df["BAIRRO"] = df["ID_OVT"].map(mapa["BAIRRO"]).fillna(df["BAIRRO"])
        df["BAIRRO_AREA"] = df["ID_OVT"].map(mapa["BAIRRO_AREA"]).fillna(df["BAIRRO_AREA"])
        # 6) matrícula dos agentes pelo nome (regra: mesmo nome em 2026 = mesma pessoa)
        df = completar_matricula(df, ref)

    ordem = ["Ano", "CICLOS", "DS", "BAIRRO", "BAIRRO_AREA", "AGENTE / MATRICULA",
             "ID_OVT", "N_OVOS", "OBS", "DATA"]
    return df[ordem].reset_index(drop=True)

In [6]:
# Importa e trata as planilhas de cada ano, usando 2026 como referência
tabelas = {}
for ano, path in path_antigos.items():
    lista = []
    for ds in DSS:
        d = tratar_ovt_antigo(path.format(ds=ds), ds=ds, ano=ano, ref=ovitrampas2026)
        print(f"{ano} DS-{ds}: {d.shape[0]} linhas, {d['ID_OVT'].nunique()} ovitrampas")
        lista.append(d)
    tabelas[ano] = pandas.concat(lista, ignore_index=True)

ovitrampas2025 = tabelas[2025]
ovitrampas2024 = tabelas[2024]
ovitrampas2024.head()

2025 DS-I: 6575 linhas, 274 ovitrampas
2025 DS-II: 10080 linhas, 420 ovitrampas
2025 DS-III: 7702 linhas, 321 ovitrampas
2025 DS-IV: 6551 linhas, 273 ovitrampas
2025 DS-V: 4488 linhas, 187 ovitrampas
2025 DS-VI: 10200 linhas, 425 ovitrampas
2025 DS-VII: 7965 linhas, 340 ovitrampas
2025 DS-VIII: 7152 linhas, 298 ovitrampas
2024 DS-I: 6491 linhas, 274 ovitrampas
2024 DS-II: 10080 linhas, 420 ovitrampas
2024 DS-III: 7702 linhas, 321 ovitrampas
2024 DS-IV: 6550 linhas, 273 ovitrampas
2024 DS-V: 4488 linhas, 187 ovitrampas
2024 DS-VI: 10199 linhas, 425 ovitrampas
2024 DS-VII: 7680 linhas, 320 ovitrampas
2024 DS-VIII: 7152 linhas, 298 ovitrampas


,Ano,CICLOS,DS,BAIRRO,BAIRRO_AREA,AGENTE / MATRICULA,ID_OVT,N_OVOS,OBS,DATA
0,2024,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,REC1001,370.0,NaN,2024-01-16
1,2024,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,REC1002,40.0,NaN,2024-01-16
2,2024,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,REC1003,57.0,NaN,2024-01-16
3,2024,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,REC1004,178.0,NaN,2024-01-16
4,2024,1,I,RECIFE ANTIGO,RECIFE ANTIGO,EMERSON MELO/77.110-0,REC1005,50.0,NaN,2024-01-16


In [7]:
# Conferências rápidas
for ano, df in tabelas.items():
    print(ano, df.shape)
    print('  Duplicados (ID_OVT, CICLOS):', df.duplicated(['ID_OVT', 'CICLOS']).sum())
    print('  Colunas só em 2026:', set(ovitrampas2026.columns) - set(df.columns))
    print('  Colunas só em', ano, ':', set(df.columns) - set(ovitrampas2026.columns))
    print('  Ovitrampas fora de 2026:', (~df['ID_OVT'].isin(ovitrampas2026['ID_OVT'])).groupby(df['ID_OVT']).first().sum())

pandas.concat(tabelas.values()).groupby(['Ano', 'DS']).agg(linhas=('ID_OVT', 'size'), ovitrampas=('ID_OVT', 'nunique'), ovos=('N_OVOS', 'sum'))

2025 (60713, 10)
  Duplicados (ID_OVT, CICLOS): 0
  Colunas só em 2026: {'OBS\nF (Fechado); E (Extraviado); R (Recusado); D (Desocupado); Rec (Recuperada)'}
  Colunas só em 2025 : {'DATA', 'OBS'}
  Ovitrampas fora de 2026: 278
2024 (60342, 10)
  Duplicados (ID_OVT, CICLOS): 0
  Colunas só em 2026: {'OBS\nF (Fechado); E (Extraviado); R (Recusado); D (Desocupado); Rec (Recuperada)'}
  Colunas só em 2024 : {'DATA', 'OBS'}
  Ovitrampas fora de 2026: 278


linhas  ovitrampas       ovos
Ano  DS                                 
2024 I       6491         274  1168510.0
     II     10080         420  2108437.0
     III     7702         321  1365588.0
     IV      6550         273  1127627.0
     V       4488         187   779675.0
     VI     10199         425  1916112.0
     VII     7680         320  1410129.0
     VIII    7152         298  1474767.0
2025 I       6575         274  1267355.0
     II     10080         420  2672602.0
     III     7702         321  1644197.0
     IV      6551         273  1352218.0
     V       4488         187   900435.0
     VI     10200         425  2225184.0
     VII     7965         340  1517854.0
     VIII    7152         298  1617636.0

## Configuração e conexões (.env)

In [8]:
import json
import os
from datetime import date, datetime, timedelta, timezone

import requests
from dotenv import dotenv_values
from pymongo import MongoClient, ReplaceOne
from requests.adapters import HTTPAdapter
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.dialects.postgresql import JSONB
from urllib3.util.retry import Retry

ENV_PATH = BASE + r"\.env"
_env = dotenv_values(ENV_PATH)


def _cfg(chave, padrao=None, obrigatorio=False):
    """Lê do .env (ou das variáveis de ambiente). Levanta erro claro se for obrigatório e faltar."""
    valor = _env.get(chave) or os.environ.get(chave) or padrao
    if obrigatorio and not valor:
        raise KeyError(f"Variável {chave} não encontrada em {ENV_PATH}")
    return valor.strip() if isinstance(valor, str) else valor

## Neon

In [9]:
NEON_SCHEMA = _cfg("NEON_SCHEMA", "public")


def _url_sqlalchemy(url):
    """O Neon entrega 'postgresql://...'; o SQLAlchemy precisa do driver explícito."""
    for prefixo in ("postgres://", "postgresql://"):
        if url.startswith(prefixo):
            return "postgresql+psycopg2://" + url[len(prefixo):]
    return url


# pool_pre_ping + connect_timeout: o Neon suspende o compute quando ocioso e a 1ª conexão pode demorar
engine = create_engine(
    _url_sqlalchemy(_cfg("NEON_DATABASE_URL", obrigatorio=True)),
    pool_pre_ping=True, pool_recycle=300, connect_args={"connect_timeout": 30},
)

with engine.begin() as con:
    if NEON_SCHEMA != "public":
        con.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{NEON_SCHEMA}"'))
    banco, versao = con.execute(text("select current_database(), version()")).fetchone()
print("Conectado ao Neon:", banco, "|", versao.split(",")[0])

Conectado ao Neon: neondb | PostgreSQL 18.6 (c021049) on aarch64-unknown-linux-gnu


In [10]:
def _snake(nome):
    """'AGENTE / MATRICULA' -> 'agente_matricula' (nomes simples, sem aspas obrigatórias no Postgres)."""
    s = unicodedata.normalize("NFKD", str(nome)).encode("ascii", "ignore").decode()
    s = re.sub(r"[^0-9a-zA-Z]+", "_", s).strip("_").lower()
    return ("c_" + s if s[:1].isdigit() else s) or "coluna"


def normalizar_colunas(df):
    novos, usados = [], set()
    for c in df.columns:
        n = base = _snake(c)
        i = 2
        while n in usados:
            n, i = f"{base}_{i}", i + 1
        usados.add(n)
        novos.append(n)
    df = df.copy()
    df.columns = novos
    return df


def enviar_neon(df, tabela, modo="replace", chave=None, indices=(), inteiros=(), lote=5000):
    
    d = normalizar_colunas(df)
    for c in inteiros:
        if c in d.columns:
            try:
                d[c] = pandas.to_numeric(d[c], errors="coerce").astype("Int64")
            except (TypeError, ValueError):
                print(f"  aviso: '{c}' tem valores não inteiros; mantido como número decimal")

    tipos = {c: JSONB for c in d.columns if d[c].map(lambda v: isinstance(v, (list, dict))).any()}
    alvo = f'"{NEON_SCHEMA}"."{tabela}"'
    opcoes = dict(schema=NEON_SCHEMA, index=False, dtype=tipos, method="multi", chunksize=lote)

    if modo in ("replace", "append"):
        d.to_sql(tabela, engine, if_exists=modo, **opcoes)

    elif modo == "upsert":
        if not chave:
            raise ValueError("modo='upsert' exige chave=[...]")
        chave = [_snake(c) for c in chave]
        d = d.drop_duplicates(chave, keep="last")            # ON CONFLICT não aceita a mesma chave 2x no lote
        col_chave = ", ".join(f'"{c}"' for c in chave)
        if not inspect(engine).has_table(tabela, schema=NEON_SCHEMA):
            d.to_sql(tabela, engine, if_exists="fail", **opcoes)
        with engine.begin() as con:
            con.execute(text(f'CREATE UNIQUE INDEX IF NOT EXISTS "{("ux_" + tabela + "_" + "_".join(chave))[:63]}" '
                             f'ON {alvo} ({col_chave})'))
        stg = f"_stg_{tabela}"[:63]
        d.to_sql(stg, engine, if_exists="replace", **opcoes)
        cols = ", ".join(f'"{c}"' for c in d.columns)
        sets = ", ".join(f'"{c}" = EXCLUDED."{c}"' for c in d.columns if c not in chave)
        acao = f"DO UPDATE SET {sets}" if sets else "DO NOTHING"
        with engine.begin() as con:
            con.execute(text(f'INSERT INTO {alvo} ({cols}) SELECT {cols} FROM "{NEON_SCHEMA}"."{stg}" '
                             f'ON CONFLICT ({col_chave}) {acao}'))
            con.execute(text(f'DROP TABLE "{NEON_SCHEMA}"."{stg}"'))
    else:
        raise ValueError("modo deve ser 'replace', 'append' ou 'upsert'")

    with engine.begin() as con:
        for cols_idx in indices:
            cols_idx = [c for c in cols_idx if c in d.columns]
            if cols_idx and not (modo == "upsert" and cols_idx == chave):   # chave já tem índice único
                nome_idx = ("ix_" + tabela + "_" + "_".join(cols_idx))[:63]
                lista = ", ".join(f'"{c}"' for c in cols_idx)
                con.execute(text(f'CREATE INDEX IF NOT EXISTS "{nome_idx}" ON {alvo} ({lista})'))
        return con.execute(text(f"SELECT COUNT(*) FROM {alvo}")).scalar()

## Ovitrampas → Neon

In [11]:
TABELAS_OVT = {
    "ovitrampas_2026": ovitrampas2026,
    "ovitrampas_2025": ovitrampas2025,
    "ovitrampas_2024": ovitrampas2024,
}

for nome, df in TABELAS_OVT.items():
    n = enviar_neon(df, nome, modo="replace",
                    inteiros=["ano", "ciclos", "n_ovos"],
                    indices=[["id_ovt", "ciclos"], ["ds"]])
    print(f"{nome}: {n} linhas no Neon (DataFrame tem {len(df)})")

ovitrampas_2026: 31455 linhas no Neon (DataFrame tem 31455)
ovitrampas_2025: 60713 linhas no Neon (DataFrame tem 60713)
ovitrampas_2024: 60342 linhas no Neon (DataFrame tem 60342)


In [12]:
# Verificação: lê de volta do Neon
print(pandas.read_sql(text("SELECT table_name FROM information_schema.tables "
                           "WHERE table_schema = :s ORDER BY 1"), engine, params={"s": NEON_SCHEMA}))
for t in TABELAS_OVT:
    print(t, pandas.read_sql(text(f'SELECT COUNT(*) AS linhas FROM "{NEON_SCHEMA}"."{t}"'), engine).iloc[0, 0])

        table_name
0  ovitrampas_2024
1  ovitrampas_2025
2  ovitrampas_2026
ovitrampas_2026 31455
ovitrampas_2025 60713
ovitrampas_2024 60342


## Coleta de APIs (funções compartilhadas pelo Neon e pelo MongoDB)

In [13]:
def _sessao():
    """Sessão HTTP com nova tentativa automática (429 e erros 5xx)."""
    s = requests.Session()
    retry = Retry(total=4, backoff_factor=1.5, status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=("GET", "POST"))
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://", HTTPAdapter(max_retries=retry))
    return s


def _resolver_segredos(obj):
    """Troca {{NOME}} pelo valor de NOME no .env (para tokens/chaves, que não devem ficar no notebook)."""
    if isinstance(obj, str):
        return re.sub(r"\{\{(\w+)\}\}", lambda m: _cfg(m.group(1), obrigatorio=True), obj)
    if isinstance(obj, dict):
        return {k: _resolver_segredos(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_resolver_segredos(v) for v in obj]
    return obj


def _extrair(obj, caminho):
    """Navega 'a.b.0.c' dentro de dicionários/listas. Sem caminho, devolve o objeto inteiro."""
    for parte in (caminho.split(".") if caminho else []):
        obj = obj[int(parte)] if isinstance(obj, list) else obj[parte]
    return obj


def colunas_para_registros(d):
    """{'time': [..], 'x': [..]} -> [{'time':.., 'x':..}, ...]  (formato em colunas, usado p.ex. pelo Open-Meteo)."""
    n = len(next(iter(d.values())))
    return [{k: v[i] for k, v in d.items()} for i in range(n)]


def buscar_api(api, sessao=None):
    """Faz a(s) requisição(ões) e devolve uma lista de dicionários (um por registro)."""
    sessao = sessao or _sessao()
    url = api["url"]
    params = _resolver_segredos(dict(api.get("params", {})))
    headers = _resolver_segredos(dict(api.get("headers", {})))
    pag = api.get("paginacao") or {}
    tipo = pag.get("tipo")
    pagina = pag.get("inicio", 1)
    registros = []

    for n_req in range(pag.get("max_paginas", 100) if tipo else 1):
        p = dict(params) if (n_req == 0 or tipo != "proximo") else None
        if tipo == "pagina":
            p[pag.get("param", "page")] = pagina
        resp = sessao.request(api.get("metodo", "GET"), url, params=p, headers=headers,
                              json=api.get("body"), timeout=api.get("timeout", 60))
        resp.raise_for_status()
        corpo = resp.json()
        dados = _extrair(corpo, api.get("caminho_dados"))
        if api.get("transformar"):
            dados = api["transformar"](dados)
        if isinstance(dados, dict):
            dados = [dados]
        if not dados:
            break
        registros.extend(dados)

        if tipo == "pagina":
            if pag.get("tamanho") and len(dados) < pag["tamanho"]:
                break
            pagina += 1
        elif tipo == "proximo":
            try:
                url = _extrair(corpo, pag.get("campo", "next"))
            except (KeyError, IndexError, TypeError):
                url = None
            if not url:
                break
        else:
            break

    extra = api.get("extra", {})
    return [{**r, **extra} for r in registros]

## APIs → Neon

In [14]:
APIS_NEON = {
    # Chuva e temperatura diárias no Recife (Open-Meteo, sem chave). Útil para cruzar com os ovos por ciclo.
    "clima_recife_diario": {
        "ativo": False,
        "url": "https://archive-api.open-meteo.com/v1/archive",
        "params": {
            "latitude": -8.05, "longitude": -34.88,
            "start_date": "2024-01-01", "end_date": str(date.today() - timedelta(days=7)),
            "daily": "precipitation_sum,temperature_2m_mean,temperature_2m_max,temperature_2m_min",
            "timezone": "America/Recife",
        },
        "caminho_dados": "daily",
        "transformar": colunas_para_registros,
        "colunas_data": ["time"],
        "renomear": {"time": "data"},
        "tabela": "clima_recife_diario",
        "modo": "upsert", "chave": ["data"],
        "indices": [],
    },

    # Modelo para uma API paginada com token (copie e ajuste)
    "exemplo_api_com_token": {
        "ativo": False,
        "url": "https://api.exemplo.gov.br/v1/registros",
        "headers": {"Authorization": "Bearer {{MINHA_API_TOKEN}}"},
        "params": {"per_page": 100},
        "paginacao": {"tipo": "pagina", "param": "page", "inicio": 1, "tamanho": 100, "max_paginas": 50},
        "caminho_dados": "results",
        "tabela": "exemplo_registros",
        "modo": "upsert", "chave": ["id"],
        "indices": [["data"]],
    },
}

In [15]:
def enviar_api_neon(nome, api):
    registros = buscar_api(api)
    if not registros:
        print(f"[vazio] {nome}: a API não devolveu registros")
        return 0
    df = pandas.json_normalize(registros, sep="_")
    for c in api.get("colunas_data", []):
        if c in df.columns:
            df[c] = pandas.to_datetime(df[c], errors="coerce")
    df = df.rename(columns=api.get("renomear", {}))
    df["ingerido_em"] = datetime.now(timezone.utc)
    return enviar_neon(df, api["tabela"], modo=api.get("modo", "replace"), chave=api.get("chave"),
                       indices=api.get("indices", ()), inteiros=api.get("inteiros", ()))


def rodar_apis(apis, enviar):
    """Roda cada API ativa; um erro em uma API não interrompe as outras."""
    resumo = {}
    for nome, api in apis.items():
        if not api.get("ativo", True):
            print(f"[pulado] {nome}")
            continue
        try:
            resumo[nome] = enviar(nome, api)
            print(f"[ok] {nome}: {resumo[nome]} registros no destino")
        except Exception as e:
            resumo[nome] = f"ERRO: {e}"
            print(f"[ERRO] {nome}: {e}")
    if not resumo:
        print("Nenhuma API ativa (defina 'ativo': True na configuração).")
    return resumo


resumo_neon = rodar_apis(APIS_NEON, enviar_api_neon)

[pulado] clima_recife_diario
[pulado] exemplo_api_com_token
Nenhuma API ativa (defina 'ativo': True na configuração).


## MongoDB

In [16]:
MONGO_USER = _cfg("MONGO_USER", obrigatorio=True)
MONGO_PASSWORD = _cfg("MONGO_PASSWORD", obrigatorio=True)
MONGO_HOST = _cfg("MONGO_HOST", obrigatorio=True)
MONGO_DB = _cfg("MONGO_DB", "gevacz")

# username/password passados como argumentos: o pymongo faz o escape sozinho
client = MongoClient(f"mongodb+srv://{MONGO_HOST}/",
                     username=MONGO_USER, password=MONGO_PASSWORD,
                     authSource="admin", serverSelectionTimeoutMS=15000)
client.admin.command("ping")
mongo_db = client[MONGO_DB]
print("Conectado ao MongoDB, banco:", MONGO_DB)

OperationFailure: bad auth : authentication failed, full error: {'ok': 0, 'errmsg': 'bad auth : authentication failed', 'code': 8000, 'codeName': 'AtlasError'}

## Ovitrampas → MongoDB

In [ ]:
def df_para_documentos(df):
    """Converte o DataFrame em lista de dicts, trocando NaN/NA/NaT por None."""
    df = df.copy()
    if "DATA" in df.columns:
        df["DATA"] = df["DATA"].astype(object)
    df = df.astype(object).where(df.notna(), None)
    docs = df.to_dict("records")
    for d in docs:
        if d.get("DATA") is not None:
            d["DATA"] = d["DATA"].to_pydatetime()   # Timestamp -> datetime
    return docs


def enviar_mongo(df, nome_colecao, lote=5000):
    col = mongo_db[nome_colecao]
    col.drop()                                  # equivale ao if_exists='replace'
    docs = df_para_documentos(df)
    for i in range(0, len(docs), lote):
        col.insert_many(docs[i:i + lote])
    col.create_index([("ID_OVT", 1), ("CICLOS", 1)])
    col.create_index("DS")
    return col.count_documents({})


for nome, df in TABELAS_OVT.items():
    n = enviar_mongo(df, nome)
    print(f"{nome}: {n} documentos (DataFrame tem {len(df)})")

## APIs → MongoDB

In [ ]:
APIS_MONGO = {
    # Alerta semanal de dengue do InfoDengue para o Recife (geocódigo IBGE 2611606)
    "infodengue_recife": {
        "ativo": False,
        "url": "https://info.dengue.mat.br/api/alertcity",
        "params": {"geocode": 2611606, "disease": "dengue", "format": "json",
                   "ew_start": 1, "ew_end": 53, "ey_start": 2024, "ey_end": date.today().year},
        "extra": {"geocode": 2611606, "doenca": "dengue"},
        "colecao": "api_infodengue_recife",
        "modo": "upsert", "chave": ["geocode", "SE"],   # a chave já cria um índice único
    },

    # Modelo (copie e ajuste)
    "exemplo_api_json": {
        "ativo": False,
        "url": "https://api.exemplo.gov.br/v1/boletins",
        "headers": {"Authorization": "Bearer {{MINHA_API_TOKEN}}"},
        "paginacao": {"tipo": "proximo", "campo": "next"},
        "caminho_dados": "results",
        "colunas_data": ["data_publicacao"],
        "colecao": "api_boletins",
        "modo": "replace",
    },
}

In [ ]:
def enviar_mongo_docs(docs, colecao, modo="replace", chave=None, indices=(), lote=5000):
    """modo: 'replace' (recria a coleção), 'append' ou 'upsert' (substitui o documento com a mesma `chave`)."""
    col = mongo_db[colecao]
    if modo == "replace":
        col.drop()
    if modo == "upsert":
        if not chave:
            raise ValueError("modo='upsert' exige chave=[...]")
        col.create_index([(c, 1) for c in chave], unique=True)
        for i in range(0, len(docs), lote):
            ops = [ReplaceOne({c: d.get(c) for c in chave}, d, upsert=True) for d in docs[i:i + lote]]
            col.bulk_write(ops, ordered=False)
    elif modo in ("replace", "append"):
        for i in range(0, len(docs), lote):
            col.insert_many(docs[i:i + lote])
    else:
        raise ValueError("modo deve ser 'replace', 'append' ou 'upsert'")
    for cols in indices:
        if modo == "upsert" and list(cols) == list(chave):
            continue                              # já existe como índice único
        col.create_index([(c, 1) for c in cols])
    return col.count_documents({})


def enviar_api_mongo(nome, api):
    registros = buscar_api(api)
    if not registros:
        print(f"[vazio] {nome}: a API não devolveu registros")
        return 0
    carimbo = datetime.now(timezone.utc)
    for r in registros:
        for c in api.get("colunas_data", []):
            if r.get(c):
                r[c] = pandas.to_datetime(r[c]).to_pydatetime()
        r["_ingerido_em"] = carimbo
    return enviar_mongo_docs(registros, api["colecao"], modo=api.get("modo", "replace"),
                             chave=api.get("chave"), indices=api.get("indices", ()))


resumo_mongo = rodar_apis(APIS_MONGO, enviar_api_mongo)

## Encerrar conexões

In [ ]:
client.close()
engine.dispose()
print("Conexões encerradas")